# AIE S4 — Warm-up 1: Optimization & Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s4-optimization-warmup.ipynb)

**Run this before Session 4.** It needs no API key and no ML-Arena
account — `torch` and `matplotlib` already ship with Colab.

Session 2 told you that training a model means minimising a loss.
This notebook does that minimisation by hand, in one variable, where
you can watch it — and then shows that `torch.optim` is performing
exactly the same arithmetic on a few thousand parameters instead of
one.

By the end you will have written:

1. gradient descent from scratch, as a `for` loop over one number
2. linear regression in PyTorch, and the five lines every training
   step is made of
3. a multi-layer perceptron on data no straight line can fit

That is the whole of Session 4's machinery, on data small enough to
plot. The challenge notebook then points it at 16,512 real rows.

---

## 0. Setup

One cell, and nothing to install. The plotting helpers are defined
here rather than imported from a package, so this notebook stands on
its own and so you can read what every figure is doing.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)


# ---- data generators -------------------------------------------------
def generate_linear_data(n_samples=100, n_features=1, noise=1.0,
                         bias=5.0, weight_scale=3.0, seed=42):
    """y = Xw + b + noise, returned as float32 tensors."""
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_samples, n_features))
    w = rng.normal(scale=weight_scale, size=(n_features, 1))
    y = X @ w + bias + rng.normal(scale=noise, size=(n_samples, 1))
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            w.astype(np.float32), float(bias))


def generate_nonlinear_data(n_samples=300, noise=0.2, seed=42):
    """A saddle: y = sin(2*x1) * cos(2*x2). No line can fit it."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, size=(n_samples, 2))
    y = np.sin(2 * X[:, 0]) * np.cos(2 * X[:, 1])
    y = y + rng.normal(scale=noise, size=n_samples)
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32).view(-1, 1))


# ---- plotting --------------------------------------------------------
def plot_function(f, theta_range=(-2, 5), ax=None):
    ax = ax or plt.subplots(figsize=(8, 4))[1]
    ts = np.linspace(*theta_range, 200)
    ax.plot(ts, [f(t) for t in ts], "-", lw=2, color="#2f6f9f")
    ax.set_xlabel("θ"); ax.set_ylabel("f(θ)"); ax.grid(alpha=0.3)
    return ax


def plot_gradient_step(theta, theta_new, grad, lr, f, theta_range=(-2, 5)):
    """One step, with the tangent whose slope is the gradient."""
    ax = plot_function(f, theta_range)
    ts = np.linspace(theta - 1, theta + 1, 20)
    ax.plot(ts, f(theta) + grad * (ts - theta), "--", color="#c1553b",
            label=f"tangent, slope {grad:.1f}")
    ax.scatter([theta], [f(theta)], s=90, color="#c1553b", zorder=5, label="before")
    ax.scatter([theta_new], [f(theta_new)], s=90, color="#2f8f4f", zorder=5, label="after")
    ax.annotate("", xy=(theta_new, f(theta_new)), xytext=(theta, f(theta)),
                arrowprops=dict(arrowstyle="->", lw=2, color="#444"))
    ax.legend(); ax.set_title(f"one step, learning rate {lr}")
    return ax


def plot_gradient_descent_1d(f, theta_history, theta_range=(-2, 5)):
    ax = plot_function(f, theta_range)
    hs = np.array(theta_history)
    ax.plot(hs, [f(t) for t in hs], "o-", ms=5, color="#c1553b", alpha=0.8)
    ax.scatter([hs[0]], [f(hs[0])], s=120, color="#c1553b", zorder=5, label="start")
    ax.scatter([hs[-1]], [f(hs[-1])], s=120, color="#2f8f4f", zorder=5, label="end")
    ax.legend(); ax.set_title(f"{len(hs) - 1} steps of gradient descent")
    return ax


def plot_loss_history(losses, title="Training loss"):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(losses, color="#2f6f9f")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax.set_title(title); ax.grid(alpha=0.3)
    return ax


def plot_predictions(X, y, y_pred, title="Predictions"):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(X, y, alpha=0.5, label="data", color="#2f6f9f")
    order = np.argsort(X.ravel())
    ax.plot(X.ravel()[order], y_pred.ravel()[order], "-", lw=2.5,
            color="#c1553b", label="model")
    ax.set_xlabel("X"); ax.set_ylabel("y"); ax.legend()
    ax.set_title(title); ax.grid(alpha=0.3)
    return ax


def plot_nonlinear_data(X, y, title="Non-linear data"):
    fig, ax = plt.subplots(figsize=(6, 5))
    s = ax.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", s=28)
    plt.colorbar(s, ax=ax, label="y")
    ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_title(title)
    return ax


def print_model_params(model, title="Parameters"):
    print(title)
    for pname, param in model.named_parameters():
        print(f"  {pname:12} shape {tuple(param.shape)}  {param.data.numpy().ravel()}")
    print(f"  total: {sum(p.numel() for p in model.parameters())} numbers")


print("setup complete —", f"torch {torch.__version__}")
print("CUDA available:", torch.cuda.is_available())

---

# Part 1 — Gradient descent, by hand

**The goal.** Find the θ that minimises a function f(θ).

**The rule.** Stand somewhere. Compute the slope. Step downhill by an
amount proportional to the slope. Repeat.

$$\theta_{new} = \theta_{old} - \eta \cdot \nabla f(\theta_{old})$$

- $\theta$ — the parameter you are choosing
- $\eta$ — the **learning rate**, how far you step
- $\nabla f(\theta)$ — the gradient, the slope of f at θ

The minus sign is the whole idea: the gradient points uphill, so you
subtract it. Everything else in this notebook is that line, applied to
more numbers at once.

## A function we can check

$f(\theta) = (3\theta - 7)^2$

A parabola, so it has exactly one minimum, and we can find it on paper
before asking a computer: the bracket is zero at $\theta = 7/3$.
Knowing the answer in advance is the point — it is how you tell that a
working optimiser is working.

In [ ]:
def f(theta):
    """The function to minimise: (3θ - 7)²."""
    return (3 * theta - 7) ** 2


def gradient_f(theta):
    """d/dθ (3θ - 7)² = 2(3θ - 7)·3 = 6(3θ - 7), by the chain rule."""
    return 6 * (3 * theta - 7)


plot_function(f)
plt.title("f(θ) = (3θ - 7)²")
plt.show()

print(f"analytical minimum: θ = 7/3 = {7 / 3:.4f}")
print(f"gradient there:     {gradient_f(7 / 3):.4f}   (zero, as it must be)")

## One step, in detail

Before the loop, watch a single step. Start at θ = 0 — the gradient
there is 6 × (0 − 7) = −42, i.e. steeply downhill to the right, so the
step should move θ up.

In [ ]:
theta = 0.0
learning_rate = 0.05

grad = gradient_f(theta)
theta_new = theta - learning_rate * grad

print(f"θ           = {theta}")
print(f"f(θ)        = {f(theta)}")
print(f"gradient    = {grad}")
print()
print(f"θ_new = θ - lr × gradient")
print(f"      = {theta} - {learning_rate} × {grad}")
print(f"      = {theta_new}")
print(f"f(θ_new)    = {f(theta_new):.4f}   (down from {f(theta)})")

plot_gradient_step(theta, theta_new, grad, learning_rate, f)
plt.show()

The loss fell from 49 to about 4.4 in one step. Now do it repeatedly.

In [ ]:
def gradient_descent(f, gradient_f, theta_init, learning_rate, n_iterations):
    """Return the whole trajectory, not just the answer — the path is
    what tells you whether the learning rate was sane."""
    theta = theta_init
    history = [theta]
    for _ in range(n_iterations):
        grad = gradient_f(theta)
        theta = theta - learning_rate * grad
        history.append(theta)
    return history


history = gradient_descent(f, gradient_f, theta_init=0.0,
                           learning_rate=0.05, n_iterations=20)

print(f"start:  θ = {history[0]:.4f}")
print(f"end:    θ = {history[-1]:.4f}")
print(f"target: θ = {7 / 3:.4f}")

plot_gradient_descent_1d(f, history)
plt.show()

Twenty steps, no calculus beyond one derivative, and it lands on the
answer. Note the steps get *shorter* as it approaches: the gradient
shrinks near the minimum, so the same learning rate takes smaller
steps. Nobody had to schedule that.

---

## Exercise 1 — the learning rate

Change `learning_rate` below and re-run. Try, in order:

| value | what to look for |
|---|---|
| `0.01` | does it arrive in 20 steps? |
| `0.05` | the baseline |
| `0.15` | faster — but look at the path |
| `0.35` | watch the final θ |

**Answer these before moving on:**

1. What does too small a learning rate cost you?
2. At what value does the path start to oscillate across the minimum
   rather than descend into it, and why does that happen?
3. `0.35` does not converge. Where does θ go, and what does that
   correspond to in a real training run?

In [ ]:
learning_rate = 0.05   # <- change me

history = gradient_descent(f, gradient_f, theta_init=0.0,
                           learning_rate=learning_rate, n_iterations=20)
print(f"lr = {learning_rate}   final θ = {history[-1]:.4f}   (target {7 / 3:.4f})")
plot_gradient_descent_1d(f, history)
plt.show()

---

## Exercise 2 — your own function

Minimise $f(\theta) = \theta^2 + 5\theta + 6$.

Work out the gradient on paper first, set it to zero to get the
analytical minimum, then check that gradient descent agrees. If they
disagree, your derivative is wrong — that is the more likely of the
two.

In [ ]:
def f2(theta):
    return theta ** 2 + 5 * theta + 6


def gradient_f2(theta):
    return 2 * theta + 5


analytical_min = -5 / 2      # where 2θ + 5 = 0
print(f"analytical minimum: θ = {analytical_min}")

history = gradient_descent(f2, gradient_f2, theta_init=5.0,
                           learning_rate=0.1, n_iterations=30)
print(f"gradient descent:   θ = {history[-1]:.4f}")

plot_gradient_descent_1d(f2, history, theta_range=(-6, 6))
plt.show()

---

# Part 2 — The same descent, in PyTorch

Linear regression is the identical problem with more parameters. You
are choosing $w$ and $b$ so that $\hat{y} = Xw + b$ minimises

$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Same rule, same minus sign — except that θ is now a vector, and you no
longer differentiate by hand. That is what PyTorch is for.

In [ ]:
X, y, true_weights, true_bias = generate_linear_data(
    n_samples=100, n_features=1, noise=1.0, bias=5.0,
    weight_scale=3.0, seed=42)

print("X", tuple(X.shape), " y", tuple(y.shape))
print(f"the answer we are hoping to recover: w = {true_weights.ravel()[0]:.3f}, b = {true_bias}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(X, y, alpha=0.6, color="#2f6f9f")
ax.set_xlabel("X"); ax.set_ylabel("y"); ax.grid(alpha=0.3)
ax.set_title("100 points from a line, plus noise")
plt.show()

## The model

`nn.Linear(1, 1)` **is** $\hat{y} = Xw + b$. It holds two numbers and
starts them at random, which is why the first predictions are nonsense.

In [ ]:
model = nn.Linear(in_features=1, out_features=1)
print_model_params(model, "Before training (random):")

with torch.no_grad():
    y_pred_initial = model(X)
    initial_loss = nn.MSELoss()(y_pred_initial, y)
print(f"\ninitial MSE: {initial_loss.item():.4f}")

plot_predictions(X.numpy(), y.numpy(), y_pred_initial.numpy(),
                 title="Before training")
plt.show()

`torch.no_grad()` says *I am only looking, do not record this for
differentiation*. Use it every time you evaluate rather than train; it
is faster and it is the habit that keeps evaluation from leaking into
the gradient.

## The loss and the optimizer

**`nn.MSELoss`** is the $\mathcal{L}$ above.

**`torch.optim.SGD`** is the update rule — the same
$\theta \leftarrow \theta - \eta \nabla$ you wrote by hand in Part 1,
applied to every parameter you hand it.

In [ ]:
model = nn.Linear(1, 1)                       # fresh start
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("the optimizer was given these tensors to update:")
for pname, param in model.named_parameters():
    print(f"  {pname} {tuple(param.shape)}")

## One training step, in detail

Every training loop you will ever write in PyTorch is these five
lines. Run the cell and read what each one changed.

In [ ]:
print("[before] parameters:")
for pname, param in model.named_parameters():
    print(f"    {pname}: {param.data.numpy().ravel()}")

print("\n[1] forward pass — y_pred = model(X)")
y_pred = model(X)
print(f"    predictions: {tuple(y_pred.shape)}")

print("\n[2] loss — how wrong is that?")
loss = loss_fn(y_pred, y)
print(f"    loss = {loss.item():.6f}")

print("\n[3] optimizer.zero_grad() — clear last step's gradients")
optimizer.zero_grad()
print("    PyTorch ACCUMULATES gradients; without this they add up")

print("\n[4] loss.backward() — differentiate the loss w.r.t. every parameter")
loss.backward()
for pname, param in model.named_parameters():
    print(f"    {pname}.grad = {param.grad.numpy().ravel()}")

print("\n[5] optimizer.step() — param -= lr * param.grad")
optimizer.step()

print("\n[after] parameters:")
for pname, param in model.named_parameters():
    print(f"    {pname}: {param.data.numpy().ravel()}")

Step 3 is the one people leave out, and it does not raise — it just
trains badly, because each step then uses the sum of every gradient so
far. If a loop of yours diverges for no visible reason, look for a
missing `zero_grad()` first.

Step 4 is autograd. You wrote `gradient_f` by hand in Part 1; here
PyTorch derived it, for both parameters, from the forward pass it
recorded.

## The full loop

Those five lines, 200 times.

In [ ]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
losses = []

for epoch in range(200):
    y_pred = model(X)                # 1. forward
    loss = loss_fn(y_pred, y)        # 2. loss
    optimizer.zero_grad()            # 3. clear
    loss.backward()                  # 4. gradients
    optimizer.step()                 # 5. update
    losses.append(loss.item())
    if (epoch + 1) % 40 == 0:
        print(f"epoch {epoch + 1:3d}: loss = {loss.item():.6f}")

print(f"\nloss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print_model_params(model, "\nLearned:")
print(f"\nTrue:  weight {true_weights.ravel()}  bias {true_bias}")

In [ ]:
plot_loss_history(losses, title="Training loss over 200 epochs")
plt.show()

with torch.no_grad():
    y_pred_final = model(X)
plot_predictions(X.numpy(), y.numpy(), y_pred_final.numpy(),
                 title="After training")
plt.show()

The learned weight and bias should be close to the ones the data was
generated from — not identical, because the data has noise in it and
the model is fitting the noise too.

Look at the loss curve: steep, then flat. The flat part is the model
telling you that more epochs will not help. That shape is the single
most useful diagnostic in deep learning, and Part 3 of the challenge
notebook plots it against a *validation* loss, which is where it
starts telling you something you cannot get any other way.

---

## Exercise 3 — learning rate and epochs, again

Now with 2 parameters instead of 1. Try `lr` in `0.001, 0.01, 0.1,
0.5` and `n_epochs` in `50, 200, 500`.

**Answer:**

1. At which learning rate does the loss become `nan`, and what has
   actually happened to the parameters when it does?
2. At `lr = 0.001`, is 200 epochs enough? How can you tell from the
   curve alone, without knowing the true answer?
3. Is there a setting that reaches a *lower* loss than `lr=0.01, 200`?
   Should there be?

In [ ]:
learning_rate = 0.01   # <- change me
n_epochs = 200         # <- and me

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
losses = []

for epoch in range(n_epochs):
    loss = loss_fn(model(X), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"lr={learning_rate}  epochs={n_epochs}  final loss={losses[-1]:.6f}")
plot_loss_history(losses)
plt.show()

---

# Part 3 — When a line is not enough

Everything above fits a straight line, and fits it well, because the
data came from a straight line.

Here is data that did not. The target is
$y = \sin(2x_1)\cos(2x_2)$ — a saddle. Colour is the value of y.

In [ ]:
X_nl, y_nl = generate_nonlinear_data(n_samples=300, noise=0.2, seed=42)
plot_nonlinear_data(X_nl.numpy(), y_nl.numpy(), title="y = sin(2·x1)·cos(2·x2) + noise")
plt.show()

Red and blue alternate in patches. No single plane over $(x_1, x_2)$
can be high in two opposite corners and low in the other two — so a
linear model cannot do better than predicting roughly the average
everywhere. Watch it fail on purpose:

In [ ]:
linear_model = nn.Linear(in_features=2, out_features=1)
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.01)
linear_losses = []

for epoch in range(500):
    loss = loss_fn(linear_model(X_nl), y_nl)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    linear_losses.append(loss.item())

print(f"linear model, 500 epochs: final loss = {linear_losses[-1]:.4f}")
print(f"variance of y (= the loss of predicting the mean): {y_nl.var().item():.4f}")

The final loss is roughly the variance of `y`. In R² terms that is
zero: 500 epochs of training bought nothing at all over predicting a
constant. The optimiser worked perfectly; the **model class** was
wrong.

## The fix: stack layers, with a non-linearity between them

A neural network is `Linear → activation → Linear → activation → …`.
The activation is load-bearing: without it, a stack of linear layers
collapses algebraically into one linear layer, and you are back where
you started. `ReLU(x) = max(0, x)` is the usual choice — it is cheap
and its gradient is 0 or 1, which does not vanish.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 32),    # 2 features  -> 32 hidden units
            nn.ReLU(),           # <- the non-linearity
            nn.Linear(32, 32),   # 32 -> 32
            nn.ReLU(),
            nn.Linear(32, 1),    # 32 -> one number
        )

    def forward(self, x):
        return self.layers(x)


nn_model = NeuralNetwork()
print(f"parameters: {sum(p.numel() for p in nn_model.parameters())}")

# Adam rather than SGD: it adapts a per-parameter step size and is what
# you will reach for by default from here on.
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.01)
nn_losses = []

for epoch in range(500):
    loss = loss_fn(nn_model(X_nl), y_nl)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    nn_losses.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1}: loss = {loss.item():.4f}")

print(f"\nneural network: {nn_losses[-1]:.4f}")
print(f"linear model:   {linear_losses[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(linear_losses, label="linear", color="#c1553b")
axes[0].plot(nn_losses, label="neural network", color="#2f6f9f")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].set_title("Same data, same optimiser, different model class")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(["linear", "neural network"],
            [linear_losses[-1], nn_losses[-1]],
            color=["#c1553b", "#2f6f9f"])
axes[1].set_ylabel("final loss"); axes[1].set_title("Final loss")
plt.tight_layout()
plt.show()

Five extra lines of model definition, and the loss falls by an order
of magnitude on data the linear model could not touch.

That is the entire argument for Session 4, and the California-housing
challenge is the same argument on real data: a straight line reaches
R² 0.576 there, and this network reaches 0.771.

---

## What you should be able to explain now

Not run — **explain**. Take each one and say it out loud:

1. Why the update rule has a minus sign in it.
2. What the learning rate trades off, and what each failure mode looks
   like on a loss curve.
3. What each of the five lines of a training step does, and what
   happens if you delete `optimizer.zero_grad()`.
4. Why `torch.no_grad()` belongs around evaluation.
5. Why removing every `nn.ReLU()` from the network in Part 3 would put
   it back at the linear model's loss.

If any of those is shaky, re-run the cell that shows it rather than
reading further — Session 4 assumes all five.

**Next:** *Warm-up 2 — CPU vs GPU*, then the challenge notebook
*AIE S4 — California Housing*.